7. Take a pandas-based script (your own, or a sample provided by your instructor) and rewrite it in
PySpark, documenting at least 3 places where the pandas approach would not scale and how Spark's
approach solves it.

In [0]:
%python
import pandas as pd
df = pd.read_csv("/Volumes/cyntexa_dev/sales/raw/sales2.csv")
df = df[df['total_amount']>1000]
df["tax"] = df["total_amount"] * 0.18
df["amount_after_tax"] = df["total_amount"] + df["tax"]       
result = df.groupby("customer_id")["total_amount"].sum()



In [0]:
%python
from pyspark.sql.functions import *

df = spark.read.csv(
    "/Volumes/cyntexa_dev/sales/raw/sales2.csv",
    header=True,
    inferSchema=True
)

result = (
    df
    .filter(col("total_amount") > 1000)
    .withColumn("tax", col("total_amount") * 0.18)
    .groupBy("customer_id")
    .agg(sum("total_amount").alias("total_revenue"))
)

result.show()

**why pandas would not scale like spark and pyspark
1.mamory limit 
-- the pandas is not suitable for processing the big data. since it's load all the data into ram first . so its fast but not suitable for operating with the large data set.
--if the machine config are 16 gb ram than pandas can't process the data more than 16gb at once while spark can process more than that.

2.large data process
--as pandas are not made for processing the large data process.cause it only load the data as per the machine configration.
--pandas are maily designed for the single machine processing

3.Group by on huge data set 
A large data set if we perform the groupbby on it than can take a lot of mamory and can become slow.

Note -> Pandas is great for small to medium datasets on one machine while Spark is designed to handle large datasets by distributing the work across multiple machines.

8. Design a partitioning/write strategy (partitionBy, target file sizes) for a table that will mostly be
queried by date range, and justify it using what you know about lazy evaluation and the physical
plan.

In [0]:
%python

df = spark.read.csv(
    "/Volumes/cyntexa_dev/sales/raw/sales2.csv",
    header=True,
    inferSchema=True
)
df.write \
    .format("delta") \
    .partitionBy("order_date") \
    .mode("overwrite") \
    .save("/Volumes/cyntexa_dev/sales/sales_table")